In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'], check=False)
# ============================================================
# Q1 ATTENTION BENCHMARK — UPGRADED VERSION
# ------------------------------------------------------------
# Main upgrades relative to the prior notebook:
# 1) stronger gold-label handling with fallback logic
# 2) distractor attack packs with family + difficulty metadata
# 3) optional multiple distractor variants per article
# 4) clean-call caching so multi-variant attacks are cheaper
# 5) truncation controls and truncation-split summaries
# 6) failure-mode classification:
#       - general degradation
#       - pro-EPU criterion drift
#       - anti-EPU criterion drift
#       - robust / neutral
#       - mixed
# 7) richer focus and confidence diagnostics
# 8) family-level and difficulty-level attack summaries
# ============================================================

import hashlib
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest
from tqdm.auto import tqdm

import kaggle_benchmarks as kbench

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600
MAX_CHARS = 6000
BOOT_N = 5000
ATTACK_VARIANTS_PER_ARTICLE = 1   # change to 2 or 3 if you want multiple distractor variants per article
OUT_DIR = Path("/kaggle/working/epu_attention_q1_upgraded_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MISSING_SENTINEL = -1

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    # "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    "qwen/qwen3-235b-a22b-instruct-2507",
]

# ---------------------------
# DISTRACTOR ATTACK PACKS
# ------------------------------------------------------------
# family: broad type of salient irrelevance
# difficulty: how semantically adjacent it is to EPU without
#             becoming true policy-uncertainty evidence
# ------------------------------------------------------------
ATTACK_PACKS = [
    {
        "attack_pack_id": "sports_easy_1",
        "family": "sports_entertainment",
        "difficulty": "easy",
        "pre_text": "A late playoff result forced coaches to reshuffle their lines, while reporters focused on injuries, locker-room reactions, and the pressure going into the next match.",
        "post_text": "Fans spent the evening debating lineup choices and replay reviews, with commentators arguing over momentum swings and the emotional impact of the loss.",
    },
    {
        "attack_pack_id": "weather_easy_1",
        "family": "weather_disruption",
        "difficulty": "easy",
        "pre_text": "Heavy rain and strong winds disrupted travel across several cities, with flooded roads, delayed buses, and repeated service alerts during the morning commute.",
        "post_text": "Airlines warned of continued delays as crews repositioned aircraft and travelers crowded terminals looking for revised departure times.",
    },
    {
        "attack_pack_id": "science_easy_1",
        "family": "science_health",
        "difficulty": "easy",
        "pre_text": "Researchers disagreed over the interpretation of a new medical study, with some calling the findings promising and others urging caution until more evidence is available.",
        "post_text": "Several labs said the headline result looked intriguing, but they emphasized that replication and a larger sample would be needed before drawing strong conclusions.",
    },
    {
        "attack_pack_id": "city_easy_1",
        "family": "urban_activity",
        "difficulty": "easy",
        "pre_text": "Downtown businesses reported uneven foot traffic during a busy weekend, as event organizers pointed to weather, parking shortages, and road closures as possible reasons.",
        "post_text": "Restaurant owners said reservations arrived in unpredictable waves, making staffing decisions harder during what is usually a reliable sales period.",
    },
    {
        "attack_pack_id": "campaign_medium_1",
        "family": "politics_elections",
        "difficulty": "medium",
        "pre_text": "Candidates spent the weekend rallying in key districts, drawing large crowds, responding to criticism, and trying to regain momentum before the next debate.",
        "post_text": "Polling analysts described the race as unusually fluid, saying campaign messaging and turnout speculation were shaping daily headlines without resolving who had the advantage.",
    },
    {
        "attack_pack_id": "markets_medium_1",
        "family": "markets_business",
        "difficulty": "medium",
        "pre_text": "Stocks swung through afternoon trading after several companies posted mixed earnings, as investors reacted to forecasts, margins, and shifting market sentiment.",
        "post_text": "Traders described the session as nervous and headline-driven, with prices jumping after analyst notes and then reversing as new rumors circulated.",
    },
    {
        "attack_pack_id": "foreign_medium_1",
        "family": "foreign_conflict",
        "difficulty": "medium",
        "pre_text": "Diplomats met for emergency talks after border tensions rose overnight, while officials issued conflicting statements about the likely direction of negotiations.",
        "post_text": "Regional observers said the situation remained volatile, with markets watching for signals but no clear agreement emerging from the meetings.",
    },
    {
        "attack_pack_id": "entertainment_medium_1",
        "family": "sports_entertainment",
        "difficulty": "medium",
        "pre_text": "A major streaming release was delayed after contract talks stalled, leading entertainment reporters to speculate about scheduling changes and audience response.",
        "post_text": "Executives insisted the company remained confident, though industry observers said the public dispute had created uncertainty around launch timing.",
    },
    {
        "attack_pack_id": "campaign_hard_1",
        "family": "politics_elections",
        "difficulty": "hard",
        "pre_text": "Campaign aides and business groups traded sharply different assessments of the race, with some warning that prolonged uncertainty could unsettle investment plans even before ballots were cast.",
        "post_text": "Analysts said firms were paying closer attention to speeches, endorsements, and shifting poll numbers, though the discussion centered on sentiment rather than any announced government decision.",
    },
    {
        "attack_pack_id": "markets_hard_1",
        "family": "markets_business",
        "difficulty": "hard",
        "pre_text": "Executives and investors spent the week debating how long volatile prices and recession fears might delay hiring and expansion plans across several industries.",
        "post_text": "Company briefings repeatedly returned to uncertainty about demand, financing conditions, and competitor reactions, even though no specific public-policy change was under discussion.",
    },
    {
        "attack_pack_id": "foreign_hard_1",
        "family": "foreign_conflict",
        "difficulty": "hard",
        "pre_text": "International headlines about tense negotiations, disrupted shipments, and uncertain commodity flows prompted new concern among firms with cross-border exposure.",
        "post_text": "Market commentators said the uncertainty was affecting expectations and planning, but the reports focused on geopolitical developments rather than domestic economic policy.",
    },
    {
        "attack_pack_id": "science_hard_1",
        "family": "science_health",
        "difficulty": "hard",
        "pre_text": "A disputed public-health report led experts, employers, and insurers to issue conflicting interpretations, leaving observers unsure how conditions might evolve over the next few quarters.",
        "post_text": "The debate generated broad uncertainty and risk-focused commentary, but it remained centered on interpretation of evidence rather than on any enacted or proposed policy measure.",
    },
    {
        "attack_pack_id": "campaign_vhard_1",
        "family": "politics_elections",
        "difficulty": "very_hard",
        "pre_text": "Business leaders said they were struggling to plan around a noisy election season, citing shifting rhetoric, legal challenges, and daily speculation about what a future administration might prioritize.",
        "post_text": "Coverage emphasized uncertainty, economic consequences, and strategic hesitation, yet the underlying story remained campaign-centered and did not report an actual policy decision or policy-uncertainty event in force.",
    },
    {
        "attack_pack_id": "markets_vhard_1",
        "family": "markets_business",
        "difficulty": "very_hard",
        "pre_text": "Investors, lenders, and exporters described an atmosphere of caution as outlooks changed rapidly and firms delayed commitments while waiting for a clearer macroeconomic picture.",
        "post_text": "Although the language sounded highly economic and uncertain, the account stayed focused on market sentiment and private-sector reactions rather than uncertainty over government policy itself.",
    },
    {
        "attack_pack_id": "foreign_vhard_1",
        "family": "foreign_conflict",
        "difficulty": "very_hard",
        "pre_text": "Merchants and analysts warned that escalating overseas tensions were clouding forecasts, complicating pricing decisions, and raising concern about future trade conditions.",
        "post_text": "The passage is deliberately policy-adjacent and economically salient, but the uncertainty comes from foreign events and expectations, not from a concrete domestic policy choice or policy debate in the target article.",
    },
    {
        "attack_pack_id": "city_vhard_1",
        "family": "urban_activity",
        "difficulty": "very_hard",
        "pre_text": "Retailers, property managers, and local lenders reported a tense business mood as closures, weak foot traffic, and uneven demand made short-term planning unusually difficult.",
        "post_text": "The narrative is economically salient and uncertainty-flavored, but it remains about local business conditions and commercial sentiment rather than uncertainty over government economic policy.",
    },
]

BAD_DISTRACTOR_TERMS = [
    "epu",
    "economic policy",
    "policy uncertainty",
    "target article",
    "classify",
    "return only",
    "json",
    "benchmark",
    "prompt",
]

for pack in ATTACK_PACKS:
    for key in ["pre_text", "post_text"]:
        low = pack[key].lower()
        assert not any(term in low for term in BAD_DISTRACTOR_TERMS), f"Bad distractor leakage in {pack['attack_pack_id']}"

# ---------------------------
# HELPERS
# ---------------------------
def stable_index(key: str, mod: int) -> int:
    h = hashlib.md5(str(key).encode("utf-8")).hexdigest()
    return int(h[:8], 16) % mod


def maybe_int(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return int(x)
    except Exception:
        try:
            return int(float(x))
        except Exception:
            return None


def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default


def safe_binary_gold(data: pd.DataFrame, preferred_col: str, fallback_col: str | None = None) -> pd.Series:
    if preferred_col in data.columns:
        s = pd.to_numeric(data[preferred_col], errors="coerce")
        if s.notna().any():
            return (s.fillna(0) >= 0.5).astype(int)
    if fallback_col is not None and fallback_col in data.columns:
        s = pd.to_numeric(data[fallback_col], errors="coerce")
        return (s.fillna(0) >= 0.5).astype(int)
    return pd.Series(np.zeros(len(data), dtype=int), index=data.index)


def smart_truncate(text: str, max_chars: int = 6000) -> tuple[str, int, int, int, float]:
    text = "" if text is None else str(text)
    orig_len = len(text)
    if orig_len <= max_chars:
        return text, orig_len, orig_len, 0, 0.0
    head = max_chars // 2
    tail = max_chars - head - 32
    truncated = text[:head] + "\n\n[...TRUNCATED...]\n\n" + text[-tail:]
    trunc_fraction = (orig_len - len(truncated)) / max(orig_len, 1)
    return truncated, orig_len, len(truncated), 1, trunc_fraction


def binary_match(pred, gold, missing_sentinel=MISSING_SENTINEL):
    gold_i = maybe_int(gold)
    pred_i = maybe_int(pred)
    if gold_i is None or gold_i == missing_sentinel:
        return None
    if pred_i is None:
        return 0.0
    return 1.0 if pred_i == gold_i else 0.0


def wilson_ci(successes: int, n: int, z: float = 1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    phat = successes / n
    denom = 1.0 + (z**2) / n
    center = (phat + (z**2) / (2 * n)) / denom
    margin = (z / denom) * math.sqrt((phat * (1 - phat) / n) + (z**2) / (4 * n**2))
    return (max(0.0, center - margin), min(1.0, center + margin))


def paired_bootstrap_diff_ci(clean_values, attacked_values, n_boot=5000, seed=42, alpha=0.05):
    a = np.asarray(clean_values, dtype=float)
    b = np.asarray(attacked_values, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    n = len(a)
    if n == 0:
        return (np.nan, np.nan)
    diffs = a - b
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = diffs[idx].mean()
    return (float(np.quantile(boot, alpha / 2)), float(np.quantile(boot, 1 - alpha / 2)))


def exact_mcnemar(clean_correct, attacked_correct):
    x = np.asarray(clean_correct, dtype=float)
    y = np.asarray(attacked_correct, dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x = x[mask].astype(int)
    y = y[mask].astype(int)
    b = int(np.sum((x == 1) & (y == 0)))
    c = int(np.sum((x == 0) & (y == 1)))
    n = b + c
    if n == 0:
        return {
            "b_hurt": b,
            "c_helped": c,
            "discordant_n": n,
            "mcnemar_p": 1.0,
            "hurt_help_ratio": np.nan,
        }
    pval = binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue
    ratio = np.inf if c == 0 and b > 0 else (b / c if c > 0 else np.nan)
    return {
        "b_hurt": b,
        "c_helped": c,
        "discordant_n": n,
        "mcnemar_p": float(pval),
        "hurt_help_ratio": ratio,
    }


def classify_distraction(drop_pp, flip_rate, pval):
    if pd.isna(drop_pp) or pd.isna(flip_rate):
        return "undetermined"
    if (drop_pp >= 0.10 and flip_rate >= 0.15 and pval < 0.05):
        return "strong distraction"
    if (drop_pp >= 0.03 and flip_rate >= 0.05 and pval < 0.05):
        return "moderate distraction"
    if drop_pp < 0.03 and flip_rate < 0.05:
        return "little distraction"
    return "mixed / borderline"


def classify_failure_mode(delta_pos, delta_neg, fp_pull_pp, fn_pull_pp, delta_overall):
    small = 0.02
    mild = 0.03
    if any(pd.isna(x) for x in [delta_pos, delta_neg, delta_overall]):
        return "undetermined"
    if abs(delta_overall) <= small and abs(delta_pos) <= small and abs(delta_neg) <= small:
        return "robust_or_neutral"
    if delta_pos <= -mild and delta_neg <= -mild:
        return "general_degradation"
    if delta_pos >= -small and delta_neg <= -mild and (not pd.isna(fp_pull_pp)) and fp_pull_pp > 0:
        return "pro_epu_criterion_drift"
    if delta_pos <= -mild and delta_neg >= -small and (not pd.isna(fn_pull_pp)) and fn_pull_pp > 0:
        return "anti_epu_criterion_drift"
    return "mixed"


def build_prompt(article_text: str, distractor_pre: str = "", distractor_post: str = "") -> str:
    blocks = []
    if distractor_pre:
        blocks.append("CONTEXT BLOCK A\n" + distractor_pre)
    blocks.append("TARGET ARTICLE START\n" + article_text + "\nTARGET ARTICLE END")
    if distractor_post:
        blocks.append("CONTEXT BLOCK B\n" + distractor_post)
    joined_text = "\n\n".join(blocks)

    return (
        "You will read multiple news-style text blocks.\n\n"
        "Your task is to classify ONLY the article between TARGET ARTICLE START and TARGET ARTICLE END.\n"
        "Ignore any surrounding context blocks, even if they sound important, political, uncertain, or economically relevant.\n\n"
        "Labeling rule:\n"
        "- EPU = 1 only if the TARGET ARTICLE discusses uncertainty about government policy, possible policy actions, policy decisions, political outcomes affecting policy, or the economic effects of policy.\n"
        "- EPU = 0 for general economics, elections, markets, war, foreign events, business conditions, or uncertainty unless the uncertainty is specifically tied to economic policy in the TARGET ARTICLE.\n\n"
        "Output rules:\n"
        "- Use evidence from the TARGET ARTICLE only.\n"
        "- If the target article does not support EPU, then who/actions/effects should usually be null.\n"
        "- mention_foreign and mainly_foreign may be 0, 1, or null if not inferable from the target article.\n"
        "- focus_excerpt must be a short excerpt copied from the TARGET ARTICLE only.\n"
        "- Return ONLY valid JSON, with no markdown, no explanation, and no code fence.\n\n"
        "Return exactly these keys:\n"
        "{\n"
        '  "epu": 0 or 1,\n'
        '  "who": 0 or 1 or null,\n'
        '  "actions": 0 or 1 or null,\n'
        '  "effects": 0 or 1 or null,\n'
        '  "mention_foreign": 0 or 1 or null,\n'
        '  "mainly_foreign": 0 or 1 or null,\n'
        '  "confidence": number between 0 and 1,\n'
        '  "focus_excerpt": "short quote from target article only"\n'
        "}\n\n"
        f"{joined_text}"
    ).strip()


def default_out():
    return {
        "epu": None,
        "who": None,
        "actions": None,
        "effects": None,
        "mention_foreign": None,
        "mainly_foreign": None,
        "confidence": 0.0,
        "focus_excerpt": "",
        "raw_text": "",
    }


def parse_llm_json(raw_text):
    out = default_out()
    out["raw_text"] = "" if raw_text is None else str(raw_text)[:3000]
    if raw_text is None:
        return out

    text = str(raw_text).strip()
    text = text.replace("```json", "").replace("```", "").strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            return out
    except Exception:
        return out

    for k in out:
        if k in obj and k != "raw_text":
            out[k] = obj[k]

    for k in ["epu", "who", "actions", "effects", "mention_foreign", "mainly_foreign"]:
        out[k] = maybe_int(out[k])

    try:
        out["confidence"] = float(out["confidence"])
    except Exception:
        out["confidence"] = 0.0
    out["confidence"] = max(0.0, min(1.0, out["confidence"]))
    out["focus_excerpt"] = "" if out["focus_excerpt"] is None else str(out["focus_excerpt"])[:250]
    return out


def model_json_call(llm, prompt: str):
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception as e:
        out = default_out()
        out["raw_text"] = f"ERROR: {str(e)[:3000]}"
        return out


def excerpt_in_target(excerpt: str, article_text: str) -> int:
    excerpt = "" if excerpt is None else str(excerpt).strip()
    article_text = "" if article_text is None else str(article_text)
    if len(excerpt) < 12:
        return 0
    return int(excerpt in article_text)


def score_composite(clean_out: dict, attacked_out: dict, row: pd.Series) -> tuple[float, dict]:
    gold_epu = maybe_int(row["gold_epu"])
    gold_who = maybe_int(row["gold_who"])
    gold_actions = maybe_int(row["gold_actions"])
    gold_effects = maybe_int(row["gold_effects"])
    gold_mention_foreign = maybe_int(row["gold_mention_foreign"])
    gold_mainly_foreign = maybe_int(row["gold_mainly_foreign"])

    score = 0.0
    denom = 0.0
    comp = {}

    m = binary_match(clean_out["epu"], gold_epu)
    comp["clean_epu_correct"] = np.nan if m is None else float(m)
    if m is not None:
        score += 0.40 * m
        denom += 0.40

    m = binary_match(attacked_out["epu"], gold_epu)
    comp["attacked_epu_correct"] = np.nan if m is None else float(m)
    if m is not None:
        score += 0.30 * m
        denom += 0.30

    clean_epu = maybe_int(clean_out["epu"])
    attacked_epu = maybe_int(attacked_out["epu"])
    stability = np.nan
    if clean_epu is not None and attacked_epu is not None:
        stability = 1.0 if clean_epu == attacked_epu else 0.0
        score += 0.10 * stability
        denom += 0.10
    comp["stable_epu_flag"] = stability

    aux_specs = [
        ("who", attacked_out["who"], gold_who),
        ("actions", attacked_out["actions"], gold_actions),
        ("effects", attacked_out["effects"], gold_effects),
        ("mention_foreign", attacked_out["mention_foreign"], gold_mention_foreign),
        ("mainly_foreign", attacked_out["mainly_foreign"], gold_mainly_foreign),
    ]
    aux_weight_each = 0.04
    aux_correct_sum = 0.0
    aux_correct_n = 0.0

    for label_name, pred, gold in aux_specs:
        m = binary_match(pred, gold)
        comp[f"attacked_{label_name}_correct"] = np.nan if m is None else float(m)
        if m is not None:
            aux_correct_sum += float(m)
            aux_correct_n += 1.0
            score += aux_weight_each * m
            denom += aux_weight_each

    comp["attacked_aux_mean"] = np.nan if aux_correct_n == 0 else aux_correct_sum / aux_correct_n
    comp["composite_score"] = 0.0 if denom <= 0 else float(score / denom)
    return comp["composite_score"], comp


def summarize_group(g: pd.DataFrame, group_name: str, group_value):
    out = {
        "group_name": group_name,
        "group_value": group_value,
        "n_rows": len(g),
    }

    clean_vec = g["clean_epu_correct"].astype(float).to_numpy()
    attacked_vec = g["attacked_epu_correct"].astype(float).to_numpy()

    clean_n = int(np.sum(~np.isnan(clean_vec)))
    attacked_n = int(np.sum(~np.isnan(attacked_vec)))

    clean_acc = float(np.nanmean(clean_vec)) if clean_n > 0 else np.nan
    attacked_acc = float(np.nanmean(attacked_vec)) if attacked_n > 0 else np.nan

    clean_k = int(np.nansum(clean_vec)) if clean_n > 0 else 0
    attacked_k = int(np.nansum(attacked_vec)) if attacked_n > 0 else 0

    clean_lo, clean_hi = wilson_ci(clean_k, clean_n)
    attacked_lo, attacked_hi = wilson_ci(attacked_k, attacked_n)

    out["clean_acc"] = clean_acc
    out["clean_acc_ci_low"] = clean_lo
    out["clean_acc_ci_high"] = clean_hi
    out["attacked_acc"] = attacked_acc
    out["attacked_acc_ci_low"] = attacked_lo
    out["attacked_acc_ci_high"] = attacked_hi

    attack_drop_pp = clean_acc - attacked_acc if clean_n > 0 and attacked_n > 0 else np.nan
    drop_lo, drop_hi = paired_bootstrap_diff_ci(clean_vec, attacked_vec, n_boot=BOOT_N, seed=SEED)
    out["attack_drop_pp"] = attack_drop_pp
    out["attack_drop_ci_low"] = drop_lo
    out["attack_drop_ci_high"] = drop_hi

    flip_n = int(g["prediction_flip"].notna().sum())
    flip_k = int(g["prediction_flip"].fillna(0).sum())
    flip_rate = flip_k / flip_n if flip_n > 0 else np.nan
    flip_lo, flip_hi = wilson_ci(flip_k, flip_n)
    out["flip_rate"] = flip_rate
    out["flip_rate_ci_low"] = flip_lo
    out["flip_rate_ci_high"] = flip_hi

    hurt_k = int(g["attack_hurt"].sum())
    help_k = int(g["attack_helped"].sum())
    hurt_rate = hurt_k / len(g) if len(g) > 0 else np.nan
    help_rate = help_k / len(g) if len(g) > 0 else np.nan
    hurt_lo, hurt_hi = wilson_ci(hurt_k, len(g))
    help_lo, help_hi = wilson_ci(help_k, len(g))
    out["attack_hurt_rate"] = hurt_rate
    out["attack_hurt_ci_low"] = hurt_lo
    out["attack_hurt_ci_high"] = hurt_hi
    out["attack_helped_rate"] = help_rate
    out["attack_helped_ci_low"] = help_lo
    out["attack_helped_ci_high"] = help_hi

    mc = exact_mcnemar(clean_vec, attacked_vec)
    out.update(mc)

    if pd.notna(clean_acc) and pd.notna(attacked_acc) and clean_acc < 1.0:
        clean_err = 1.0 - clean_acc
        attacked_err = 1.0 - attacked_acc
        out["relative_error_increase"] = (attacked_err / clean_err - 1.0) if clean_err > 0 else np.nan
    else:
        out["relative_error_increase"] = np.nan

    gold0 = g[g["gold_epu"] == 0].copy()
    gold1 = g[g["gold_epu"] == 1].copy()

    if len(gold0) > 0:
        clean_fp_rate = gold0["clean_fp"].mean()
        attacked_fp_rate = gold0["attacked_fp"].mean()
        clean_acc_gold0 = gold0["clean_epu_correct"].mean()
        attacked_acc_gold0 = gold0["attacked_epu_correct"].mean()
        out["clean_fp_rate_gold0"] = clean_fp_rate
        out["attacked_fp_rate_gold0"] = attacked_fp_rate
        out["fp_pull_pp"] = attacked_fp_rate - clean_fp_rate
        out["clean_acc_gold0"] = clean_acc_gold0
        out["attacked_acc_gold0"] = attacked_acc_gold0
        out["delta_acc_gold0"] = attacked_acc_gold0 - clean_acc_gold0
    else:
        out["clean_fp_rate_gold0"] = np.nan
        out["attacked_fp_rate_gold0"] = np.nan
        out["fp_pull_pp"] = np.nan
        out["clean_acc_gold0"] = np.nan
        out["attacked_acc_gold0"] = np.nan
        out["delta_acc_gold0"] = np.nan

    if len(gold1) > 0:
        clean_fn_rate = gold1["clean_fn"].mean()
        attacked_fn_rate = gold1["attacked_fn"].mean()
        clean_acc_gold1 = gold1["clean_epu_correct"].mean()
        attacked_acc_gold1 = gold1["attacked_epu_correct"].mean()
        out["clean_fn_rate_gold1"] = clean_fn_rate
        out["attacked_fn_rate_gold1"] = attacked_fn_rate
        out["fn_pull_pp"] = attacked_fn_rate - clean_fn_rate
        out["clean_acc_gold1"] = clean_acc_gold1
        out["attacked_acc_gold1"] = attacked_acc_gold1
        out["delta_acc_gold1"] = attacked_acc_gold1 - clean_acc_gold1
    else:
        out["clean_fn_rate_gold1"] = np.nan
        out["attacked_fn_rate_gold1"] = np.nan
        out["fn_pull_pp"] = np.nan
        out["clean_acc_gold1"] = np.nan
        out["attacked_acc_gold1"] = np.nan
        out["delta_acc_gold1"] = np.nan

    out["delta_overall"] = out["attacked_acc"] - out["clean_acc"] if pd.notna(out["clean_acc"]) and pd.notna(out["attacked_acc"]) else np.nan
    out["stable_rate"] = g["stable_epu_flag"].mean()
    out["composite_score_mean"] = g["composite_score"].mean()
    out["attacked_aux_mean"] = g["attacked_aux_mean"].mean()
    out["clean_confidence_mean"] = g["clean_confidence"].mean()
    out["attacked_confidence_mean"] = g["attacked_confidence"].mean()
    out["delta_confidence"] = out["attacked_confidence_mean"] - out["clean_confidence_mean"]
    out["attacked_confidence_harmed_rows"] = g.loc[g["attack_hurt"] == 1, "attacked_confidence"].mean()
    out["attacked_confidence_helped_rows"] = g.loc[g["attack_helped"] == 1, "attacked_confidence"].mean()
    out["clean_excerpt_in_target_rate"] = g["clean_excerpt_in_target"].mean()
    out["attacked_excerpt_in_target_rate"] = g["attacked_excerpt_in_target"].mean()
    out["excerpt_shift_off_target_rate"] = g["excerpt_shift_off_target"].mean()
    out["wrong_but_attacked_excerpt_in_target_rate"] = g.loc[g["attacked_epu_correct"] == 0, "attacked_excerpt_in_target"].mean()
    out["was_truncated_rate"] = g["was_truncated"].mean() if "was_truncated" in g.columns else np.nan

    out["distraction_level"] = classify_distraction(out["attack_drop_pp"], out["flip_rate"], out["mcnemar_p"])
    out["failure_mode"] = classify_failure_mode(
        out["delta_acc_gold1"],
        out["delta_acc_gold0"],
        out["fp_pull_pp"],
        out["fn_pull_pp"],
        out["delta_overall"],
    )
    return out


# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]
for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)
clean = df.loc[mask].copy()

clean["article_key"] = clean["article_key"].astype(str)
missing_key_mask = clean["article_key"].isin(["nan", "None", ""])
if missing_key_mask.any():
    clean.loc[missing_key_mask, "article_key"] = [f"generated_key_{i}" for i in range(missing_key_mask.sum())]

clean["gold_epu"] = pd.to_numeric(clean["gold_epu"], errors="coerce").fillna(0).astype(int)
clean["gold_mention_foreign"] = safe_binary_gold(clean, "gold_mention_foreign", "mention_foreign")
clean["gold_mainly_foreign"] = safe_binary_gold(clean, "gold_mainly_foreign", "mainly_foreign")
clean["gold_who"] = pd.to_numeric(clean.get("gold_who", np.nan), errors="coerce").fillna(MISSING_SENTINEL).astype(int)
clean["gold_actions"] = pd.to_numeric(clean.get("gold_actions", np.nan), errors="coerce").fillna(MISSING_SENTINEL).astype(int)
clean["gold_effects"] = pd.to_numeric(clean.get("gold_effects", np.nan), errors="coerce").fillna(MISSING_SENTINEL).astype(int)
clean["disagreement_flag"] = pd.to_numeric(clean.get("disagreement_flag", 0), errors="coerce").fillna(0).astype(int)
clean["newspaper"] = clean.get("newspaper", "").fillna("").astype(str)
clean["year"] = pd.to_numeric(clean.get("year", np.nan), errors="coerce")
clean["content_len"] = pd.to_numeric(clean.get("content_len", np.nan), errors="coerce")

clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
print("EPU class balance in cleaned pool:")
print(clean["gold_epu"].value_counts(dropna=False).sort_index())
print("Disagreement balance in cleaned pool:")
print(clean["disagreement_flag"].value_counts(dropna=False).sort_index())

# ---------------------------
# BALANCED SAMPLE
# ---------------------------
def balanced_sample(data: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()
    if len(pos) == 0 or len(neg) == 0:
        return data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)
    n_pos = min(len(pos), n // 2)
    n_neg = min(len(neg), n - n_pos)
    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed + 1)
    return (
        pd.concat([pos_s, neg_s], axis=0)
        .sample(frac=1, random_state=seed + 2)
        .reset_index(drop=True)
    )

pilot = balanced_sample(clean, N_ROWS, SEED).copy()
print("\nPilot class balance:")
print(pilot["gold_epu"].value_counts().sort_index())
print("\nPilot disagreement balance:")
print(pilot["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# TRUNCATION + ATTACK VARIANTS
# ---------------------------
trunc_info = pilot["content"].astype(str).apply(lambda x: pd.Series(smart_truncate(x, MAX_CHARS), index=["content_for_eval", "orig_content_len", "eval_content_len", "was_truncated", "truncation_fraction"]))
pilot = pd.concat([pilot.reset_index(drop=True), trunc_info.reset_index(drop=True)], axis=1)


def choose_attack_pack(article_key: str, variant_idx: int) -> dict:
    idx = stable_index(f"{article_key}__attack_variant_{variant_idx}", len(ATTACK_PACKS))
    return ATTACK_PACKS[idx]


expanded_rows = []
for row in pilot.to_dict(orient="records"):
    for variant_idx in range(ATTACK_VARIANTS_PER_ARTICLE):
        pack = choose_attack_pack(row["article_key"], variant_idx)
        rec = dict(row)
        rec["attack_variant_id"] = variant_idx
        rec["attack_pack_id"] = pack["attack_pack_id"]
        rec["attack_family"] = pack["family"]
        rec["attack_difficulty"] = pack["difficulty"]
        rec["distractor_pre"] = pack["pre_text"]
        rec["distractor_post"] = pack["post_text"]
        rec["eval_row_id"] = f"{row['article_key']}__v{variant_idx}__{pack['attack_pack_id']}"
        expanded_rows.append(rec)

eval_df = pd.DataFrame(expanded_rows)

# save pilot used
pilot_path = OUT_DIR / "pilot_rows_q1_upgraded.csv"
pilot.to_csv(pilot_path, index=False)
attack_bank_path = OUT_DIR / "attack_pack_bank_q1.csv"
pd.DataFrame(ATTACK_PACKS).to_csv(attack_bank_path, index=False)
expanded_eval_path = OUT_DIR / "eval_rows_with_attack_variants_q1.csv"
eval_df.to_csv(expanded_eval_path, index=False)

# ---------------------------
# MANUAL Q1 RUN
# ---------------------------
records = []
model_handles = {name: kbench.llms[name] for name in MODEL_NAMES}

for model_name in MODEL_NAMES:
    llm = model_handles[model_name]
    print(f"\nRunning {model_name} ...")
    clean_cache = {}

    for row in tqdm(eval_df.to_dict(orient="records"), total=len(eval_df), desc=model_name):
        article_key = row["article_key"]
        if article_key not in clean_cache:
            clean_prompt = build_prompt(row["content_for_eval"])
            clean_cache[article_key] = model_json_call(llm, clean_prompt)
        clean_out = clean_cache[article_key]

        attacked_prompt = build_prompt(row["content_for_eval"], row["distractor_pre"], row["distractor_post"])
        attacked_out = model_json_call(llm, attacked_prompt)

        composite_score, comp = score_composite(clean_out, attacked_out, pd.Series(row))

        clean_pred = maybe_int(clean_out["epu"])
        attacked_pred = maybe_int(attacked_out["epu"])
        gold_epu = maybe_int(row["gold_epu"])

        clean_correct = comp["clean_epu_correct"]
        attacked_correct = comp["attacked_epu_correct"]
        stable_flag = comp["stable_epu_flag"]

        attack_hurt = int(clean_correct == 1.0 and attacked_correct == 0.0) if not pd.isna(clean_correct) and not pd.isna(attacked_correct) else 0
        attack_helped = int(clean_correct == 0.0 and attacked_correct == 1.0) if not pd.isna(clean_correct) and not pd.isna(attacked_correct) else 0
        prediction_flip = int((clean_pred is not None) and (attacked_pred is not None) and (clean_pred != attacked_pred))

        clean_fp = int(gold_epu == 0 and clean_pred == 1) if clean_pred is not None else 0
        attacked_fp = int(gold_epu == 0 and attacked_pred == 1) if attacked_pred is not None else 0
        clean_fn = int(gold_epu == 1 and clean_pred == 0) if clean_pred is not None else 0
        attacked_fn = int(gold_epu == 1 and attacked_pred == 0) if attacked_pred is not None else 0

        clean_excerpt_in_target = excerpt_in_target(clean_out["focus_excerpt"], row["content_for_eval"])
        attacked_excerpt_in_target = excerpt_in_target(attacked_out["focus_excerpt"], row["content_for_eval"])
        excerpt_shift_off_target = int(clean_excerpt_in_target == 1 and attacked_excerpt_in_target == 0)

        rec = {
            "llm_name": model_name,
            "eval_row_id": row["eval_row_id"],
            "article_key": row["article_key"],
            "attack_variant_id": row["attack_variant_id"],
            "attack_pack_id": row["attack_pack_id"],
            "attack_family": row["attack_family"],
            "attack_difficulty": row["attack_difficulty"],
            "gold_epu": row["gold_epu"],
            "gold_who": row["gold_who"],
            "gold_actions": row["gold_actions"],
            "gold_effects": row["gold_effects"],
            "gold_mention_foreign": row["gold_mention_foreign"],
            "gold_mainly_foreign": row["gold_mainly_foreign"],
            "disagreement_flag": row["disagreement_flag"],
            "newspaper": row.get("newspaper", ""),
            "year": row.get("year", np.nan),
            "content_len": row.get("content_len", np.nan),
            "orig_content_len": row.get("orig_content_len", np.nan),
            "eval_content_len": row.get("eval_content_len", np.nan),
            "was_truncated": row.get("was_truncated", 0),
            "truncation_fraction": row.get("truncation_fraction", 0.0),
            "content_for_eval": row["content_for_eval"],
            "distractor_pre": row["distractor_pre"],
            "distractor_post": row["distractor_post"],
            "clean_epu_pred": clean_pred,
            "attacked_epu_pred": attacked_pred,
            "clean_epu_correct": clean_correct,
            "attacked_epu_correct": attacked_correct,
            "stable_epu_flag": stable_flag,
            "prediction_flip": prediction_flip,
            "attack_hurt": attack_hurt,
            "attack_helped": attack_helped,
            "clean_fp": clean_fp,
            "attacked_fp": attacked_fp,
            "clean_fn": clean_fn,
            "attacked_fn": attacked_fn,
            "clean_who_pred": maybe_int(clean_out["who"]),
            "clean_actions_pred": maybe_int(clean_out["actions"]),
            "clean_effects_pred": maybe_int(clean_out["effects"]),
            "clean_mention_foreign_pred": maybe_int(clean_out["mention_foreign"]),
            "clean_mainly_foreign_pred": maybe_int(clean_out["mainly_foreign"]),
            "attacked_who_pred": maybe_int(attacked_out["who"]),
            "attacked_actions_pred": maybe_int(attacked_out["actions"]),
            "attacked_effects_pred": maybe_int(attacked_out["effects"]),
            "attacked_mention_foreign_pred": maybe_int(attacked_out["mention_foreign"]),
            "attacked_mainly_foreign_pred": maybe_int(attacked_out["mainly_foreign"]),
            "attacked_who_correct": comp["attacked_who_correct"],
            "attacked_actions_correct": comp["attacked_actions_correct"],
            "attacked_effects_correct": comp["attacked_effects_correct"],
            "attacked_mention_foreign_correct": comp["attacked_mention_foreign_correct"],
            "attacked_mainly_foreign_correct": comp["attacked_mainly_foreign_correct"],
            "attacked_aux_mean": comp["attacked_aux_mean"],
            "clean_confidence": safe_float(clean_out["confidence"]),
            "attacked_confidence": safe_float(attacked_out["confidence"]),
            "clean_focus_excerpt": clean_out["focus_excerpt"],
            "attacked_focus_excerpt": attacked_out["focus_excerpt"],
            "clean_excerpt_in_target": clean_excerpt_in_target,
            "attacked_excerpt_in_target": attacked_excerpt_in_target,
            "excerpt_shift_off_target": excerpt_shift_off_target,
            "clean_raw_json": clean_out["raw_text"],
            "attacked_raw_json": attacked_out["raw_text"],
            "composite_score": composite_score,
        }
        records.append(rec)

detailed_df = pd.DataFrame(records)
raw_path = OUT_DIR / "q1_results_detailed_upgraded.csv"
detailed_df.to_csv(raw_path, index=False)

# ---------------------------
# SUMMARY TABLES
# ---------------------------
summary_overall = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "overall", "all"))
        for model_name, g in detailed_df.groupby("llm_name", sort=False)
    ]
).sort_values(["attack_drop_pp", "flip_rate"], ascending=[False, False])
summary_overall_path = OUT_DIR / "summary_overall_q1_upgraded.csv"
summary_overall.to_csv(summary_overall_path, index=False)

summary_by_disagreement = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "disagreement_flag", int(flag)))
        for (model_name, flag), g in detailed_df.groupby(["llm_name", "disagreement_flag"], sort=False)
    ]
)
summary_by_disagreement_path = OUT_DIR / "summary_by_disagreement_q1_upgraded.csv"
summary_by_disagreement.to_csv(summary_by_disagreement_path, index=False)

summary_by_gold_epu = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "gold_epu", int(flag)))
        for (model_name, flag), g in detailed_df.groupby(["llm_name", "gold_epu"], sort=False)
    ]
)
summary_by_gold_epu_path = OUT_DIR / "summary_by_gold_epu_q1_upgraded.csv"
summary_by_gold_epu.to_csv(summary_by_gold_epu_path, index=False)

summary_by_family = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "attack_family", family))
        for (model_name, family), g in detailed_df.groupby(["llm_name", "attack_family"], sort=False)
    ]
)
summary_by_family_path = OUT_DIR / "summary_by_attack_family_q1_upgraded.csv"
summary_by_family.to_csv(summary_by_family_path, index=False)

summary_by_difficulty = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "attack_difficulty", difficulty))
        for (model_name, difficulty), g in detailed_df.groupby(["llm_name", "attack_difficulty"], sort=False)
    ]
)
summary_by_difficulty_path = OUT_DIR / "summary_by_attack_difficulty_q1_upgraded.csv"
summary_by_difficulty.to_csv(summary_by_difficulty_path, index=False)

summary_by_truncation = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "was_truncated", int(flag)))
        for (model_name, flag), g in detailed_df.groupby(["llm_name", "was_truncated"], sort=False)
    ]
)
summary_by_truncation_path = OUT_DIR / "summary_by_truncation_q1_upgraded.csv"
summary_by_truncation.to_csv(summary_by_truncation_path, index=False)

# ---------------------------
# TRANSITIONS + HARD CASES
# ---------------------------
transition_rows = []
for (model_name, gold_epu), g in detailed_df.groupby(["llm_name", "gold_epu"], sort=False):
    tmp = (
        g.assign(
            clean_label=g["clean_epu_pred"].fillna(-9).astype(int),
            attacked_label=g["attacked_epu_pred"].fillna(-9).astype(int),
        )
        .groupby(["clean_label", "attacked_label"])
        .size()
        .reset_index(name="count")
    )
    tmp["llm_name"] = model_name
    tmp["gold_epu"] = gold_epu
    transition_rows.append(tmp)
transition_df = pd.concat(transition_rows, ignore_index=True)
transition_path = OUT_DIR / "transition_counts_clean_to_attacked_q1_upgraded.csv"
transition_df.to_csv(transition_path, index=False)

harmed_rows = (
    detailed_df.groupby(["article_key", "gold_epu", "disagreement_flag"], as_index=False)
    .agg(
        harmed_models=("attack_hurt", "sum"),
        helped_models=("attack_helped", "sum"),
        mean_composite=("composite_score", "mean"),
        mean_clean_correct=("clean_epu_correct", "mean"),
        mean_attacked_correct=("attacked_epu_correct", "mean"),
        was_truncated=("was_truncated", "max"),
        attack_families=("attack_family", lambda x: " | ".join(sorted(set(map(str, x))))),
        attack_difficulties=("attack_difficulty", lambda x: " | ".join(sorted(set(map(str, x))))),
        content_for_eval=("content_for_eval", "first"),
        distractor_pre=("distractor_pre", "first"),
        distractor_post=("distractor_post", "first"),
    )
    .sort_values(["harmed_models", "mean_attacked_correct"], ascending=[False, True])
)
harmed_rows_path = OUT_DIR / "most_harmed_rows_shared_q1_upgraded.csv"
harmed_rows.to_csv(harmed_rows_path, index=False)

worst_attack_cases = (
    detailed_df[detailed_df["attack_hurt"] == 1]
    .sort_values(["llm_name", "attacked_confidence", "composite_score"], ascending=[True, False, True])
)
worst_attack_cases_path = OUT_DIR / "worst_attack_failures_by_model_q1_upgraded.csv"
worst_attack_cases.to_csv(worst_attack_cases_path, index=False)

# ---------------------------
# MANIFEST
# ---------------------------
manifest = pd.DataFrame({
    "file_name": [
        raw_path.name,
        summary_overall_path.name,
        summary_by_disagreement_path.name,
        summary_by_gold_epu_path.name,
        summary_by_family_path.name,
        summary_by_difficulty_path.name,
        summary_by_truncation_path.name,
        transition_path.name,
        harmed_rows_path.name,
        worst_attack_cases_path.name,
        pilot_path.name,
        attack_bank_path.name,
        expanded_eval_path.name,
    ],
    "description": [
        "Row-level clean and attacked predictions, attack metadata, truncation flags, confidences, excerpts, and composite score.",
        "Main model table with clean accuracy, attacked accuracy, attack drop, flip rate, FP/FN pull, distraction level, and failure mode.",
        "Scientific proof metrics split by disagreement_flag.",
        "Scientific proof metrics split by gold_epu.",
        "Attack-effect summaries split by distractor family.",
        "Attack-effect summaries split by distractor difficulty.",
        "Attack-effect summaries split by truncation status.",
        "Transition counts from clean prediction to attacked prediction, by model and gold_epu.",
        "Rows harmed by distractors across multiple models, for qualitative evidence.",
        "Model-specific rows where distractors changed a clean-correct prediction into an attacked-wrong prediction.",
        "The 600-row pilot article sample used for the run.",
        "Full attack pack bank with family and difficulty labels.",
        "Expanded evaluation rows after attaching attack variants to pilot articles.",
    ]
})
manifest_path = OUT_DIR / "manifest_q1_upgraded.csv"
manifest.to_csv(manifest_path, index=False)

print("\nSaved files:")
for p in [
    raw_path,
    summary_overall_path,
    summary_by_disagreement_path,
    summary_by_gold_epu_path,
    summary_by_family_path,
    summary_by_difficulty_path,
    summary_by_truncation_path,
    transition_path,
    harmed_rows_path,
    worst_attack_cases_path,
    pilot_path,
    attack_bank_path,
    expanded_eval_path,
    manifest_path,
]:
    print(p)

print("\nOverall proof table:")
display(summary_overall)
